# Experiment 10 — Nonlinear (MLP) probe on CLS embeddings

A 2-layer MLP probe (hidden $\in$ {128, 256, 512} selected by validation AUC,
ReLU, dropout $\in$ {0.0, 0.1, 0.3}, Adam lr=1e-3, up to 40 epochs with
early stopping on validation loss, patience 5) is fit on the cached CLS
embeddings for every (model, dataset, perturbation) cell in which the linear
CLS probe yielded AUC $\leq$ 0.55. Seeds $\in$ {0, 1, 2}; results
aggregated as mean $\pm$ sd. A linear probe that fails at chance could
reflect either absence of the information or linear inaccessibility; an MLP
probe distinguishes those cases.


In [ ]:
import os, sys, warnings
from pathlib import Path
import pandas as pd

ROOT = Path(os.environ.get('V4_WORK_DIR',
    '/home/saptpurk/embeddings-noise-eliminators/v4_work'))
DATASET = os.environ.get('DATASET', 'nih')
MODELS = (os.environ.get('MODELS_TO_RUN',
    'raddino,dinov2,biomedclip,dinov3,medsiglip').split(','))
RUN_TAG = os.environ.get('RUN_TAG', 'gpu0')

def _collect_linear_auc():
    recs = []
    for f in sorted(ROOT.glob(f'v4_exp02_geometric_{DATASET}/*_gpu*.parquet')):
        d = pd.read_parquet(f); d['exp'] = 'exp02'
        d['artifact'] = d.get('pattern', '?'); recs.append(d)
    for f in sorted(ROOT.glob(f'v4_exp03_iso_motion_blur_{DATASET}/*_gpu*.parquet')):
        d = pd.read_parquet(f); d['exp'] = 'exp03'
        d['artifact'] = 'iso_blur_p' + d['patch_size'].astype(str); recs.append(d)
    return pd.concat(recs, ignore_index=True) if recs else pd.DataFrame()

lin = _collect_linear_auc()
out_dir = ROOT / f'v4_exp10_mlp_probe_{DATASET}'
out_dir.mkdir(parents=True, exist_ok=True)

if lin.empty:
    pd.DataFrame({'status': ['inputs_absent']}).to_parquet(
        out_dir / f'exp10_{DATASET}_manifest_{RUN_TAG}.parquet', index=False)
else:
    lin = lin[lin['model'].isin(MODELS)]
    weak = lin[lin['auc'] <= 0.55][['exp', 'model', 'artifact']].drop_duplicates()
    sys.path.insert(0, str(Path('..').resolve()))
    try:
        from common.probing import fit_mlp_probe_from_cache
    except Exception as e:
        warnings.warn(f'fit_mlp_probe_from_cache import failed: {e}')
        fit_mlp_probe_from_cache = None

    rows = []
    if fit_mlp_probe_from_cache is not None:
        for _, r in weak.iterrows():
            for seed in (0, 1, 2):
                try:
                    res = fit_mlp_probe_from_cache(
                        exp=r['exp'], model=r['model'], artifact=r['artifact'],
                        dataset=DATASET, seed=seed)
                    rows.append({**r.to_dict(), 'seed': seed, **res})
                except FileNotFoundError:
                    continue
    out = pd.DataFrame(rows)
    out_path = out_dir / f'exp10_{DATASET}_mlp_probe_{RUN_TAG}.parquet'
    out.to_parquet(out_path, index=False)
    print(f'wrote {len(out)} rows -> {out_path}')
